# Practical 1 - Introduction to the UR3e

 Before starting, make sure you completed Part 1 of the practical.

> :information_source: Check whether you have a working network connection to the robot and it is in **Remote Control**.

## Part 2 - Controlling the robot from Python

### 2.1 - Getting started with `airo_robots` 

The `airo_robots` is a python package that we wrote with several Python classes to make controlling robot hardware simpler.

<div class="alert alert-block alert-info">  <b>Task 1</b>: Fill in the robot's ip address and create the robot object.</div>

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True) # This makes the output look nicer

from airo_robots.manipulators.hardware.ur_rtde import URrtde

ip_address = "10.42.0.162"  # fill in the robot IP

robot = URrtde(ip_address, URrtde.UR3E_CONFIG)

This class has several methods that read data from the robot.

In [ ]:
q = robot.get_joint_configuration()
q

In [ ]:
X_B_TCP = robot.get_tcp_pose()
X_B_TCP

## 2.2 - Moving the robot

This class also provides methods that are very similar to the ones you used in Polyscope to move the robot. 
For example:
* `MoveJ` → `robot.move_to_tcp_pose()`
* `MoveL` → `robot.move_linear_to_tcp_pose()`
* `MoveJ with joint angles` → `robot.move_to_joint_configuration()`

To start, let's try to move the robot to a joint configuration.

<div class="alert alert-block alert-info">  <b>Task 2</b>: Keep the emergency button close and run the cell below to move the robot.</div>
<div class="alert alert-block alert-danger"> From now on one person should keep an eye on the robot's movement while being close to the emergency button</div>

In [ ]:
q_straight = [0, -np.pi/2, 0, -np.pi/2, 0, 0]  # also possible: joint_angles = np.deg2rad([0, -90, 0, -90, 0, 0])

robot.move_to_joint_configuration(q_straight).wait()

If the command succeeded, the robot should now be standing up straight. However this is still quite far from where the robot will be working.

<div class="alert alert-block alert-info">  <b>Task 3</b>: Choose 6 joint angles for a comfortable "home" configuration and move the robot to it. </div>

You can do this by putting the robot in freedrive through python by running the command of the next cell. We advise putting the robot via Python in freedrive instead of using the teaching pendant. The reason for doing so is that you have to restart this notebook everytime you put the robot back in "local" mode. This is a current shortcoming caused by using notebooks. 

In [ ]:
robot.rtde_control.teachMode()

You can now move the robot around. Put it in a home position and run the following cell to disengage freedrive.

In [ ]:
robot.rtde_control.endTeachMode()

Now write down the joint values from the teaching pendant or get them via our `robot` object.

In [ ]:
q_home = []

robot.move_to_joint_configuration(q_home).wait()

> :information_source: **Note the `.wait()` command.** Under the hood, any robot movement is asynchronous. If we wish to build a sequential pipeline, we must explicitly wait for a move command to finish, before we send another move command.

## 2.3 - TCP movements
As seen in PolyScope, it is also possible to move the robot by specifying a target pose for the TCP.
This is often referred to as **task-space control**, as opposed to the **joint-space control** in the previous cells.

In [ ]:
X_B_TCP = robot.get_tcp_pose()
X_B_TCP

<div class="alert alert-block alert-info">  <b>Task 4</b>: modify the TCP pose to move the 10 cm in the x direction w.r.t. base. Note that the base frame axis are labeled on the robot's base.</div>

In [ ]:
X_B_TCP_2 = X_B_TCP.copy()

# TODO modify pose here

robot.move_to_tcp_pose(X_B_TCP_2).wait()
robot.move_to_tcp_pose(X_B_TCP).wait() # move back to the original pose

## 2.4 Orientation of the TCP

Changing the position of the TCP is fairly simple. However, changing the orientation is a bit more tricky and there are several ways to do it.

In the cell below, we show how you can construct a 3x3 rotation matrix by stacking together 3 orthonormal vectors.

In [ ]:
X = np.array([1, 0, 0])
Z = np.array([0, -1, 0])
Y = np.cross(Z, X)

R_B_TCP = np.column_stack([X, Y, Z]) # Orientation TCP in base frame
R_B_TCP

In [ ]:
p_B_TCP = np.array([0.3, -0.1, 0.2]) # Position of TCP in base frame

X_B_TCP_3 = np.identity(4)
X_B_TCP_3[:3, :3] = R_B_TCP
X_B_TCP_3[:3, 3] = p_B_TCP
X_B_TCP_3

<div class="alert alert-block alert-info">  <b>Task 5</b>: Discuss with your team members how you think the robot will move when you run the cell below. :speech_balloon: When you are confident, run the cell and see if you were right.</div>

In [ ]:
robot.move_to_tcp_pose(X_B_TCP_3).wait()

The orientation the robot is in right now might not be ideal for picking up pipettes.
It's up to you to create a new top-down orientation that is more suitable.

<div class="alert alert-block alert-info">  <b>Task 6</b>: Create a top-down orientation for the robot and move it there.</div>

In [ ]:
X_B_TCP_4 = X_B_TCP_3.copy()

# TODO complete the function below (you are allowed to implement it differently if you want)
def top_down_orientation() -> np.ndarray:
    X = []
    Y = []
    Z = []
    R = np.column_stack([X, Y, Z])
    return R


R_B_TCP = top_down_orientation()
X_B_TCP_4[:3, :3] = R_B_TCP

robot.move_to_tcp_pose(X_B_TCP_4).wait()

## 2.5 - Picking pipette tips

<img align="left" height="400" src="https://i.imgur.com/IL0Emrg.png">
<BR CLEAR="all">

Now you be ready to implement the pipette picking program you designed in Part 1 in Python.
As an exercise we want you express your program in terms of a **world** reference frame.
You can choose where you place this world frame, as long as it is *different from the robot's base frame*.

<div class="alert alert-block alert-info">  <b>Task 7</b>: Measure and make a sketch of the robot's environment and write down the poses you will need. Also consider how the robot should move between these poses.</div>

<div class="alert alert-block alert-info">  <b>Task 8</b>: Implement below a program to pick up all the pipette tips.</div>

In [ ]:
# TODO: implement the pipette picking and dropping here

## 2.6 - Force mode

Another useful feature of the UR robots is called "force mode".
In force mode, you can specify in how much force you want the robot to apply in a certain direction.
If the robot feels less force than desired, it will move in that direction.

Below is an example where the robot try to push down with a force of 5 N.

<div class="alert alert-block alert-info">  <b>Task 9</b>: Run the cell below and see what happens. Try to carefully push the robot back up.</div>

In [ ]:
import time

task_frame = [0, 0, 0, 0, 0, 0] # Base frame
compliant_axes = [1, 1, 1, 0, 0, 0] # Position compliant, but orientation locked
force_type = 2
limits = [0.2, 0.2, 0.2, 1, 1, 1] # Ensures robot won't move more than 20 cm/s 

desired_force = [0, 0, -5, 0, 0, 0] # The force the robot will try to achieve -5 N in the z direction

# Turn on force mode for 5 seconds
robot.rtde_control.zeroFtSensor() # Calibrates the force measurement
robot.rtde_control.forceMode(task_frame, compliant_axes, desired_force, force_type, limits)

for i in range(10):
    print(np.array(robot.rtde_receive.getActualTCPForce())[:3])
    time.sleep(1)

robot.rtde_control.forceModeStop()

> :thought_balloon: What could be the advantage of using force mode as opposed to position control?

<div class="alert alert-block alert-info">  <b>Task 10</b>: Experiment with force mode and use it to pick up the pipettes.</div>

# The end

Congratulations! You have now completed the first practical of the course.

A few questions to think about:

* Why could it be preferable to define a starting pose in joint space, rather than tool space?
* Which positions are unreachable by the robot and why? Make a drawing.
* Why do we need intermediate poses? How could we avoid having to manually define such poses?
* What could make the picking of the pipette tips more robust?

Before shutting down the robot, please use freedrive to put it in a safe position (away from objects and do not let any links of robots rest outside the table so people do not collide against the robot when passing).

In [ ]:
robot.rtde_control.teachMode()

In [ ]:
robot.rtde_control.endTeachMode()